# Отбор на вакансию Стажер в отдел разработки биоинформатического программного обеспечения

**Молекулярным генетикам часто требуется собрать последовательность ДНК с нуля.
Синтез длинных последовательностей работает неэффективно, поэтому прибегают к
сборке последовательностей из олигонуклеотидов — коротких ДНК — с помощью ПЦР.
Таким образом, возникает задача дизайна олигонуклеотидов: по данной
последовательности ДНК сгенерировать схему разбиения её на подходящие кусочки.
Эту задачу в нашем департаменте решают при помощи скрипта, который принимает на
вход полную последовательность ДНК и ряд параметров, среди которых есть длина
генерируемых олигонуклеотидов и ограничение на минимальный возможный
размер перекрытия, то есть участка, на котором два соседних олигонуклеотида
связываются друг с другом во время ПЦР.
Ниже Вы найдете упрощенную реализацию дизайна олигов на языке Haskell.\
Ваша задача:**

## 1) Описать, что именно она делает, и какой смысл в проверяемых ею ограничениях с точки зрения лабораторного процесса.

In [ ]:
{-# LANGUAGE RecordWildCards #-}
module OligoSplitter (
splitOligos,
maxTempDiff,
SplittingConfig(..)
) where
import Data.List (sortOn)
maxTempDiff :: Double
maxTempDiff = 5.0 # максимальная разница в температуре между отжигом двух перекрывающихся участков
data SplittingConfig = SplittingConfig # Задается разбиение через фиксированную длину олигонуклеотида и min длину перекрытия между соседними олигонуклеотидами
{ oligoSize :: Int
, minOverlap :: Int
} deriving (Show)
type DNA = String
type Position = Int
type OverlapTemp = Double
data Oligo = Oligo # Структура олигонуклеотида (плавление нуклеотида, начало нуклеотида, конце нуклеотида)
{ oligoStart :: Position
, oligoEnd :: Position
, oligoTemp :: OverlapTemp
} deriving (Show, Eq)
type SplittingResult = [Oligo] 
-- Упрощённая модель: температура = базовая + 4*G/C + 2*A/T
countTemp :: DNA -> Double
countTemp sequ = baseTemp + gcContribution + atContribution 
where
baseTemp = 15.0
gcCount = length $ filter (`elem` "GCgc") sequ
atCount = length $ filter (`elem` "ATat") sequ
gcContribution = 4.0 * fromIntegral gcCount
atContribution = 2.0 * fromIntegral atCount
getSubstring :: DNA -> Position -> Position -> DNA
getSubstring sequ start end = take (end - start + 1) $ drop start sequ
calcOverlapTemp :: DNA -> Position -> Position -> OverlapTemp # расчет температуры плавления для перекрытия по упрозенной формуле
calcOverlapTemp sequ nextStart prevEnd = countTemp overlapSeq
where
overlapSeq = getSubstring sequ nextStart prevEnd # само перекрытие
checkAllTempsInRange :: [Oligo] -> Bool
checkAllTempsInRange [] = True
checkAllTempsInRange [_] = True
checkAllTempsInRange oligs =
all inRange tempPairs
where
temps = map oligoTemp oligs
tempPairs = [(temps !! i, temps !! j) | i <- [0..length temps - 1],
j <- [i+1..length temps - 1]]
inRange (t1, t2) = abs (t1 - t2) <= maxTempDiff # Ограничение в разнице между температурами перекрытий
generateNextPositions :: SplittingConfig -> Position -> Position -> [Position] # Перебор возможных позиций для начала следующего олигонуклеотида
generateNextPositions SplittingConfig{..} lastOligoEnd seqLen =
filter validPosition [minNext .. maxNext]
where
maxOverlap = oligoSize `div` 2 # максимальная длина перекрытия - длина олиго/2 
minNext = max 0 (lastOligoEnd - maxOverlap + 1)
maxNext = lastOligoEnd - minOverlap + 1
validPosition p =
let nextEnd = p + oligoSize - 1
in p >= 0 && nextEnd < seqLen
generateAllSplittings :: SplittingConfig -> DNA -> Position -> [SplittingResult];# рекурсивный перебор всех допустимых положений каждого следующего нуклеотида
generateAllSplittings splitConfig sequ lastPos
| lastPos >= seqLen - 1 = [[]]
| otherwise = do
nextStart <- generateNextPositions splitConfig lastPos seqLen # начало следующего олиго
let nextEnd = nextStart + oligoLen - 1
let overlapTemp = calcOverlapTemp sequ nextStart lastPos # вычисление температуры
let currentOligo = Oligo nextStart nextEnd overlapTemp
rest <- generateAllSplittings splitConfig sequ nextEnd # рекурсивно достариваем остальные
let fullSplitting = currentOligo : rest
if checkAllTempsInRange fullSplitting # оставляем только корректные варианты
then [fullSplitting]
else []
where
oligoLen = oligoSize splitConfig
seqLen = length sequ
splitAll :: SplittingConfig -> DNA -> [SplittingResult]# собираем найденные разбиения
splitAll splitConfig sequ = do
let firstEnd = oligoSize splitConfig - 1
rest <- generateAllSplittings splitConfig sequ firstEnd
let firstOligo = Oligo 0 firstEnd 0
return $ firstOligo : rest
splitOligos :: SplittingConfig -> DNA -> Maybe SplittingResult
splitOligos splitConfig sequ =
case validSplittings of
[] -> Nothing
splits -> Just $ sortOn (length) splits !! 0 # выбираем с минимальным числом олигонуклеотидов
where
validSplittings = splitAll splitConfig sequ

### Биологический смысл

#### Минимальное перекрытие

_overlap>=minOverlap_

При сборке конструкции соседние олиги должны гибридизоваться друг с другом. Если перекрытия короткие:
1. гибридизация нестабильна
2. вероятность ошибок растет
3. эффективность ПЦР падает

#### Максимальное перекрытие

_overlap<=oligoSize/2_

Слишком большие перекрытия:
1. Увеличивают стоимость синтеза
2. повышают вероятность втор. структур
3. уменьшают долю уникальной последовательности каждого олиго
4. Термодинамически олиги становятся похожи друг на друга

#### Близость температур плавления

_|Tm1 - Tm2| <= 5_

Нужна одна температура отжига, иначе нельзя подобрать оптимальную: одни участки не будут связываться, а другие будут давать неспецифичное связывание

## 2) Найти места, написанные неэффективно.

### 1. Подсчет температуры
Подсчет температуры написан крайне неэффективно. Каждый раз температура плавления для участка считается итеративно, хотя достаточно поддерживать массив префиксных сумм, чтобы ускорить решение до константного времени. 
### 2. Рекурсия
На языке Python есть ограничение на глубину рекурсии в 1000 вызовов из-за чего программа не сможет обрабатывать сложные случаи с большой длиной строки.
### 3. Прверка разности температур
Проверка на разности температур сравнивает каждое перекрытие с каждым - подходят ли конкретно 2 сплита, что крайне неэффективно.
### 4. Проверка на корректность выбранной разбивки происходит только после ее генерации.
Условие на схожесть температуры отжига между перекрытиями олигов проверяются только после генерации решения. Из-за чего в самом начале мы можем сразу получить неподходящее решение и ресурсы продолжат тратиться на него.

## 3)Предложить свою реализацию на Python, C++ или Haskell, лишенную этих недостатков: возвращающую корректные разбиения и работающую быстрее.

In [198]:
from dataclasses import dataclass
from typing import List, Optional

max_temp_diff = 5.0

dp = {}
parent = {}

# -------------------------
# DATA STRUCTURES
# -------------------------

@dataclass
class SplittingConfig:
    """
    Configuration for oligo splitting.

    Attributes:
        oligo_size: fixed length of each oligo
        min_overlap: minimal allowed overlap between consecutive oligos
    """
    oligo_size: int
    min_overlap: int


@dataclass
class Oligo:
    """
    Represents a single oligo segment.

    Attributes:
        oligo_start: start index in DNA sequence
        oligo_end: end index in DNA sequence
        oligo_temp: computed overlap temperature metric
    """
    oligo_start: int
    oligo_end: int
    oligo_temp: float


# -------------------------
# PREFIX SUM (FAST TEMP CALC)
# -------------------------

def build_prefix(seq):
    """
    Builds prefix sum array for fast GC/AT-based temperature computation.

    Each G/C adds +4, each A/T adds +2, base is 15.
    """
    pref = [15.0]

    for n in seq:
        if n in "GCgc":
            pref.append(pref[-1] + 4)
        else:
            pref.append(pref[-1] + 2)

    return pref


def calc_overlap_temp(pref, start, end):
    """
    Computes temperature contribution for substring [start:end]
    using prefix sums in O(1).
    """
    return pref[end + 1] - pref[start]


# -------------------------
# VALID POSITION GENERATION
# -------------------------

def generate_next_positions(config, last_oligo_end, seq_len):
    """
    Generates all valid next oligo start positions
    based on overlap constraints.
    """
    max_overlap = config.oligo_size // 2

    min_start = max(
        0,
        last_oligo_end - max_overlap + 1
    )

    max_start = min(
        last_oligo_end - config.min_overlap + 1,
        seq_len - config.oligo_size
    )

    if min_start > max_start:
        return []

    return range(min_start, max_start + 1)


# -------------------------
# PATH RECONSTRUCTION
# -------------------------

def solve(start_state):
    """
    Reconstructs oligo path from DP parent pointers.

    Returns list of Oligo objects in forward order.
    """
    result = []
    state = start_state

    while state in parent:
        next_start, next_end, tm, next_state = parent[state]

        result.append(
            Oligo(
                oligo_start=next_start,
                oligo_end=next_end,
                oligo_temp=tm
            )
        )

        state = next_state

    return result


# -------------------------
# DP CORE (DFS + MEMOIZATION)
# -------------------------

def generate_all_splittings(
    config: SplittingConfig,
    seq: str,
    start_last_pos: int,
    pref,
    min_tm=None,
    max_tm=None
):
    """
    Main DP engine.

    Finds optimal sequence of oligos satisfying:
        - fixed oligo size
        - overlap constraints
        - temperature range constraint (max_temp_diff)

    Uses iterative DFS + memoization.
    """
    seq_len = len(seq)
    oligo_len = config.oligo_size

    # stack item: (position, min_temp, max_temp, is_expanded)
    stack = [(start_last_pos, min_tm, max_tm, False)]

    start_state = (start_last_pos, min_tm, max_tm)

    while stack:

        last_pos, min_tm, max_tm, is_expanded = stack.pop()
        key = (last_pos, min_tm, max_tm)

        if key in dp:
            continue

        # -------------------------
        # BASE CASES
        # -------------------------
        if last_pos > seq_len - 1:
            dp[key] = None
            continue

        if last_pos == seq_len - 1:
            dp[key] = 0
            continue

        # -------------------------
        # EXPAND NODE
        # -------------------------
        if not is_expanded:
            stack.append((last_pos, min_tm, max_tm, True))

            for next_start in generate_next_positions(
                config,
                last_pos,
                seq_len
            ):
                next_end = next_start + oligo_len - 1

                overlap_temp = calc_overlap_temp(pref, next_start, last_pos)

                # initialize or update temperature 
                if min_tm is None:
                    new_min = overlap_temp
                    new_max = overlap_temp
                else:
                    new_min = min(min_tm, overlap_temp)
                    new_max = max(max_tm, overlap_temp)

                # pruning condition
                if new_max - new_min > max_temp_diff:
                    continue

                child_state = (next_end, new_min, new_max)
                stack.append((*child_state, False))

        # -------------------------
        #  DP TRANSITION
        # -------------------------
        else:
            best = float("inf")

            for next_start in generate_next_positions(
                config,
                last_pos,
                seq_len
            ):
                next_end = next_start + oligo_len - 1

                overlap_temp = calc_overlap_temp(pref, next_start, last_pos)

                if min_tm is None:
                    new_min = overlap_temp
                    new_max = overlap_temp
                else:
                    new_min = min(min_tm, overlap_temp)
                    new_max = max(max_tm, overlap_temp)

                if new_max - new_min > max_temp_diff:
                    continue

                child = (next_end, new_min, new_max)

                rest = dp.get(child)
                if rest is None:
                    continue

                candidate = rest + 1

                if candidate < best:
                    best = candidate

                    parent[key] = (
                        next_start,
                        next_end,
                        overlap_temp,
                        child
                    )

            dp[key] = None if best == float("inf") else best

    return dp.get(start_state)


# -------------------------
# RUNNING BLOCK
# -------------------------

def split_all(config, seq, pref):
    """
    Runs DP and reconstructs full oligo solution.
    """
    first_end = config.oligo_size - 1

    start_state = (first_end, None, None)

    best_length = generate_all_splittings(
        config,
        seq,
        first_end,
        pref
    )

    if best_length is None:
        return None

    rest = solve(start_state)

    first_oligo = Oligo(
        oligo_start=0,
        oligo_end=first_end,
        oligo_temp=0.0
    )

    return [first_oligo] + rest


def split_oligos(config: SplittingConfig, seq: str):
    """
    Main entry point.

    Returns optimal list of oligos or None if no solution exists.
    """
    dp.clear()
    parent.clear()

    pref = build_prefix(seq)

    return split_all(config, seq, pref)


# -------------------------
# FASTA OUTPUT
# -------------------------

def write_fasta(result, seq, filename):
    """
    Writes oligos into FASTA file format.

    Each oligo is extracted from original sequence.
    """
    if result is None:
        print("No solution")
        return

    with open('./outputs/'+filename, "w") as ans:
        for i, oligo in enumerate(result):
            ans.write(f">Seq_{i}\n")
            ans.write(seq[oligo.oligo_start:oligo.oligo_end + 1] + "\n")

    print("Success!!")

# Тестирование

### 1

In [199]:
cfg = SplittingConfig(
    oligo_size=10,
    min_overlap=3
)

seq = (
    "GGGGGGGGGGGGGGGGGGGG"
    "AAAAAAAAAAAAAAAAAAAA"
    "GGGGGGGGGGGGGGGGGGGG"
)

In [200]:
result = split_oligos(cfg, seq)
write_fasta(result, seq, 'ans1.fasta')

Success!!


### 2

In [201]:
test_cases = [

    (
        "ATGC" * 20,
        SplittingConfig(
            oligo_size=12,
            min_overlap=4
        ),
        True
    ),

    (
        "ATATATATATATATATATATATATATATATATATATATATATATATAT",
        SplittingConfig(
            oligo_size=12,
            min_overlap=4
        ),
        True
    ),

    (
        "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC",
        SplittingConfig(
            oligo_size=12,
            min_overlap=4
        ),
        True
    ),

    (
        "ATGCATGCATGC",
        SplittingConfig(
            oligo_size=12,
            min_overlap=4
        ),
        True
    ),

    (
        "ATGCATGC",
        SplittingConfig(
            oligo_size=12,
            min_overlap=4
        ),
        False
    ),

    (
        "GGGGGGGGGGGGGGGGGGGG"
        "AAAAAAAAAAAAAAAAAAAA"
        "GGGGGGGGGGGGGGGGGGGG"
        "AAAAAAAAAAAAAAAAAAAA",
        SplittingConfig(
            oligo_size=20,
            min_overlap=10
        ),
        False
    ),

    (
        "GGGGAAAAGGGGAAAAGGGGAAAAGGGGAAAA",
        SplittingConfig(
            oligo_size=16,
            min_overlap=8
        ),
        True
    ),

    (
        "ATGC" * 15,
        SplittingConfig(
            oligo_size=8,
            min_overlap=4
        ),
        True
    ),

    (
        "ATGC" * 20,
        SplittingConfig(
            oligo_size=16,
            min_overlap=7
        ),
        True
    ),

    (
        "ATGC" * 20,
        SplittingConfig(
            oligo_size=16,
            min_overlap=8
        ),
        True
    ),

]

In [202]:
for i, case in enumerate(test_cases):
    seq, cfg, ans = case
    result = split_oligos(cfg, seq)
    if (result is None) ^ ans:
        write_fasta(result, seq, f'case_{i}.fasta')

Success!!
Success!!
Success!!
Success!!
No solution
No solution
Success!!
Success!!
Success!!
Success!!


### 3

In [203]:
import random
seq = ''.join(random.choices("ATGC", k=200000))
config = SplittingConfig(
    oligo_size=30,
    min_overlap=10
)

In [204]:
%%time
result = split_oligos(config,seq)
write_fasta(result, seq, f'long.fasta')

Success!!
CPU times: user 6.13 s, sys: 191 ms, total: 6.32 s
Wall time: 6.32 s
